<div dir="rtl" align="right">

# משימה 2 — הוכחת הרצה של B ו־C

המחברת מפעילה רק את B ו־C, בזו אחר זו, על תרחיש שיקגו S01 הקיים. אין הרחבות או סוכנים נוספים. מגבלות: 3 סבבים בסך הכול ו־270 שניות לכל מערכת. מספר קריאות המודל עשוי להיות גדול משלוש.

לפני ההפעלה: בחלונית **Secrets** (סמל המפתח) ודאו שקיים `GEMINI_API_KEY` והפעילו **Notebook access** למחברת זו. אין להכניס מפתח לתא קוד או לשיחה. בחרו **Runtime → Run all**; אין צורך ב־GPU. בסיום הורידו את קובץ ה־ZIP. זו הרצה מול Gemini בחשבון שלכם, בכפוף לזמינות המודל ולמכסה שלו.

המודל שהוגדר כאן תואם לברירת המחדל הקיימת בממשק. זמינותו בחשבון עדיין טעונה אימות בהרצה. `task2_complete=true` יתקבל רק אם שתי המערכות החזירו תוכנית תקינה, נצפו תשובות מודל אמיתיות, הבקשה לא השתנתה וכל המגבלות נשמרו. כשל אינו נמחק ואינו מוצג כהצלחה. מחברת זו לא דוחפת אוטומטית תוצרים ל־GitHub; העלו את ה־ZIP לשיחה כדי לבדוק ולשמור את הראיות ב־main.

</div>


In [ ]:
from pathlib import Path
import subprocess
import sys
import uuid

PROJECT = Path("/content") / ("Travel_Agent_task2_" + uuid.uuid4().hex[:8])
subprocess.run([
    "git", "clone", "--depth", "1", "--branch", "main",
    "https://github.com/Noven2155/Travel_Agent.git", str(PROJECT)
], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-task2.txt"
], cwd=PROJECT, check=True)
print("Source and dependencies are ready.")


In [ ]:
import datetime as dt
import json
import os
from google.colab import userdata

MODEL = "gemini/gemini-3.1-flash-lite"
try:
    api_key = userdata.get("GEMINI_API_KEY")
except Exception:
    raise RuntimeError("Enable Notebook access for GEMINI_API_KEY in Colab Secrets.") from None
if not isinstance(api_key, str) or not api_key.strip():
    raise RuntimeError("GEMINI_API_KEY is empty in Colab Secrets.")

RUN_DIR = PROJECT / "evidence/task2" / dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
model_env = dict(os.environ)
model_env.pop("GOOGLE_API_KEY", None)
model_env.update({
    "GEMINI_API_KEY": api_key.strip(), "LLM_MODEL": MODEL,
    "LLM_TEMPERATURE": "0", "LLM_MAX_TOKENS": "8192", "LLM_SEED": "42",
    "LLM_TIMEOUT_S": "60", "WEATHER_MODE": "fixture",
    "CREWAI_DISABLE_TELEMETRY": "true", "OTEL_SDK_DISABLED": "true",
})
del api_key
try:
    completed = subprocess.run([
        sys.executable, "scripts/prove_bc.py", "--output", str(RUN_DIR)
    ], cwd=PROJECT, env=model_env)
finally:
    model_env.clear()
if (RUN_DIR / "summary.json").exists():
    print(json.dumps(json.loads((RUN_DIR / "summary.json").read_text()), ensure_ascii=False, indent=2))
print("Runner exit code:", completed.returncode)


In [ ]:
import shutil
from google.colab import files

if not RUN_DIR.is_dir():
    raise RuntimeError("No evidence directory was created; inspect the previous cell.")
archive = shutil.make_archive(str(RUN_DIR), "zip", root_dir=RUN_DIR.parent, base_dir=RUN_DIR.name)
print("Evidence archive is ready. Upload this ZIP to the task 2 conversation.")
files.download(archive)
